<a href="https://colab.research.google.com/github/Summy121/121/blob/main/%E9%BA%BB%E8%BE%A3%E5%AE%87%E5%AE%993_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌶️ 麻辣宇宙 3.1｜親手抽辣椒神秘卡的 AI 點餐店
**課堂作業：Tool Use AI Agent + Gradio**

- 🍜 正常點餐與 🎲 命運盲盒（盲盒預設 NT$125），自由點菜及套餐加點。
- 🍱 單人套餐 NT$139（含一杯茶），雙人套餐 NT$299（含兩杯茶），飲料升級收差額。
- 🃏 結帳小計滿 NT$250 才解鎖三張神秘卡；顧客**自己選一張**才真正執行抽獎工具。
- 🎁 9/8/7/6/5 折或未中獎；每筆相同訂單只能抽一次；折扣後顯示實付。
- 🛠️ AI 呼叫組單、計價工具；抽獎由顧客觸發 Python 工具，兩種來源有明確工具日誌。

價格、抽獎均為**課堂模擬**。

**2.7 介面修正：** 套餐飲料欄位固定可見，避免前端切換事件失敗導致選項消失；Gradio 6 的 CSS/theme 移至 launch()。若出現 Unexpected token <，關閉先前的 gradio.live 舊分頁，只使用最新執行輸出的網址。

**3.1 更新：** 抽獎仍是真隨機，每張新訂單中獎機率 80%，未中獎機率 20%；沒有保證每次中獎。每筆訂單只可抽一次。


**3.1 改進：** 主食價格調整，新增白飯；正常點餐顯示蔥花、香菜選項；命運盲盒專屬「不要的食材」；盲盒用剩餘預算挑菜（已先扣套餐、飲料、指定食材），不擅自刪除顧客必選品項，超支則提示。


**3.1 更新**：盲盒不再自行選菜／填份數，改由 AI 工具依「搭配方向、預算、完整忌口清單」選出盡可能豐富的品項；盲盒也能選擇要不要蔥花、香菜。正常點餐仍可自行選菜。


## Step 1｜安裝依賴及設定 API 金鑰
在 Colab 左側「🔑 密鑰」新增 **`Groq`**，填入自己的 Groq API Key，並打開筆記本存取。不要公開金鑰。

In [4]:
!pip -q install "aisuite[groq]" "gradio>=6,<7"

In [5]:
import os, json, re, random, hashlib
import aisuite as ai
import gradio as gr
from google.colab import userdata

key = userdata.get("Groq")
if not key:
    raise ValueError("請到 Colab 左側 🔑 密鑰設定 Groq，並開啟筆記本存取權")
os.environ["GROQ_API_KEY"] = key
client = ai.Client()
provider, model = "groq", "openai/gpt-oss-120b"
print("✅ Groq Key 已載入（不會顯示金鑰）")

✅ Groq Key 已載入（不會顯示金鑰）


## Step 2｜菜單和工具
- `get_menu`：查菜價；`get_drink_options`：查飲料。
- `build_order`：建立套餐、盲盒及加點。
- `calculate_price`：套餐優惠＋加菜＋飲料升級差額。
- `check_budget`：檢查盲盒預算。
- **`draw_surprise`**：只有顧客按下三張神秘卡之一、且滿 NT$250 時才呼叫。

In [6]:
MENU = {
    "牛肉片": (55,"肉類"), "豬肉片": (45,"肉類"), "雞肉片": (45,"肉類"), "羊肉片": (60,"肉類"),
    "高麗菜": (25,"蔬菜"), "青江菜": (25,"蔬菜"), "金針菇": (25,"蔬菜"), "玉米筍": (30,"蔬菜"), "海帶": (20,"蔬菜"), "娃娃菜": (30,"蔬菜"),
    "豆腐": (20,"豆製品"), "豆皮": (25,"豆製品"), "凍豆腐": (25,"豆製品"),
    "魚丸": (30,"火鍋料"), "貢丸": (30,"火鍋料"), "蟹肉棒": (25,"火鍋料"),
    "王子麵": (15,"主食"), "冬粉": (15,"主食"), "年糕": (30,"主食"), "烏龍麵": (35,"主食"), "白飯": (15,"主食"),
    "鴨血": (25,"特色"), "毛肚": (65,"特色")
}
CATEGORIES = ["肉類","蔬菜","豆製品","火鍋料","主食","特色"]
DIRECTIONS = ["完全隨機","肉肉狂歡","蔬菜滿滿","澱粉快樂","均衡營養感"]
# 套餐固定優惠價，已含飲料；套餐內食材及飲料不得重複計價。
# 調整王子麵價格後：單人套餐食材 115 + 基本紅茶 25 = 140；雙人餐食材 305 + 兩杯紅茶 50 = 355
COMBOS = {
    "單人套餐": {"items":{"牛肉片":1,"高麗菜":1,"豆腐":1,"王子麵":1},"price":139,"drinks_included":1},
    "雙人套餐": {"items":{"牛肉片":2,"高麗菜":2,"金針菇":1,"豆皮":2,"烏龍麵":2},"price":299,"drinks_included":2},
}
DRINKS = {
    "紅茶":25, "綠茶":25, "無糖烏龍茶":30, "冬瓜茶":30,
    "酸梅湯":35, "可樂":35, "奶茶":45,
}
BASE_DRINK_PRICE = 25
DRINK_OPTIONS = ["紅茶","綠茶","無糖烏龍茶","冬瓜茶","酸梅湯","可樂","奶茶"]

def get_drink_options():
    return [{"飲料":name,"單價":price} for name,price in DRINKS.items()]

def normalize_counts(obj):
    output={}
    for item, qty in (obj or {}).items():
        if item not in MENU: continue
        try: qty=int(qty)
        except (TypeError,ValueError): continue
        if qty>0: output[item]=min(qty,20)
    return output

def get_menu():
    return [{"食材":name,"類別":cat,"單價":price} for name,(price,cat) in MENU.items()]

def category_pool(direction):
    mapping={"肉肉狂歡":["肉類","特色"], "蔬菜滿滿":["蔬菜","豆製品"],
             "澱粉快樂":["主食","豆製品"], "均衡營養感":CATEGORIES}
    cats=mapping.get(direction,CATEGORIES)
    return [k for k,v in MENU.items() if v[1] in cats]

def choose_blind_ingredients(direction, excluded, reserved, remaining_budget, persons, seed):
    """優先不同食材，在預算內多選幾種；方向偏好加權，最後才補份數。"""
    rng=random.Random(int(seed))
    available=[k for k in MENU if k not in set(excluded) and k not in set(reserved)]
    if not available:return {}
    # 無預算上限時仍設合理份數，避免一次無限制加菜。
    budget_limit=remaining_budget
    max_unique=min(len(available),max(8,min(22,int(persons)*6+3)))
    tastes={
        "肉肉狂歡":{"肉類":5,"特色":2.6,"蔬菜":.75,"豆製品":1.0,"主食":.75,"火鍋料":1.2},
        "蔬菜滿滿":{"蔬菜":5,"豆製品":2.5,"肉類":.65,"主食":.9,"特色":.6,"火鍋料":.7},
        "澱粉快樂":{"主食":6,"豆製品":1.5,"蔬菜":1.0,"肉類":.7,"特色":.6,"火鍋料":.8},
        "均衡營養感":{"蔬菜":2.5,"肉類":2.2,"主食":2.3,"豆製品":2.0,"特色":1.2,"火鍋料":1.1}
    }
    priority=tastes.get(direction,{cat:1.0 for cat in CATEGORIES})
    remaining=available[:]
    result={};spent=0;used=set(MENU[k][1] for k in reserved)
    # 限額時為了多選種類，每次會保留低價品項的機會，避免一份貴肉耗盡額度。
    for i in range(max_unique):
        feasible=[k for k in remaining if budget_limit is None or spent+MENU[k][0]<=budget_limit]
        if not feasible:break
        affordable_count=max(1,min(len(feasible),max_unique-i))
        scores=[]
        for k in feasible:
            price,cat=MENU[k]
            w=priority.get(cat,1.0)
            if cat not in used:w*=1.35
            if budget_limit is not None:
                funds=budget_limit-spent
                # 把省下的錢留給更多不同菜品，而不是一開始選最昂貴的。
                min_price=min(MENU[z][0] for z in feasible)
                if price>min_price:w*=(min_price/price)**1.25
                if funds-price < min_price and len(result)<2:w*=.3
            scores.append(max(.001,w))
        picked=rng.choices(feasible,weights=scores,k=1)[0]
        remaining.remove(picked)
        result[picked]=1;spent+=MENU[picked][0];used.add(MENU[picked][1])
    # 如果剩下金額還能加點，就優先補符合方向且便宜的食材份數。
    if budget_limit is not None and result:
        for _ in range(min(12,max(0,int(persons)*3))):
            feasible=[k for k in result if spent+MENU[k][0]<=budget_limit]
            if not feasible:break
            scored=[priority.get(MENU[k][1],1)/MENU[k][0] for k in feasible]
            k=rng.choices(feasible,weights=scored,k=1)[0]
            result[k]+=1;spent+=MENU[k][0]
    return result

def build_order(mode, persons, selected, direction, avoid, package, drink1, drink2, seed,
                single_drink="不點飲料", budget=None):
    persons=max(1,min(12,int(persons)))
    avoid=set(avoid or []) if mode=="命運盲盒" else set()
    extras={} if mode=="命運盲盒" else normalize_counts(selected)
    package_items={};warnings=[]
    if package in COMBOS:
        original=COMBOS[package]["items"]
        package_items={k:v for k,v in original.items() if k not in avoid}
        removed=[k for k in original if k in avoid]
        if removed:warnings.append("套餐內已排除忌口食材："+"、".join(removed)+"；套餐價格不變，請確認是否仍需此套餐。")
    else:package="不選套餐"
    drinks=[]
    if package in COMBOS:
        requested=[drink1] if package=="單人套餐" else [drink1,drink2]
        drinks=[d if d in DRINKS else "紅茶" for d in requested]
    elif single_drink in DRINKS:
        drinks=[single_drink]
    # 盲盒不接收手動選菜，只先扣套餐與飲料，再把剩餘預算用於不同食材。
    fixed_cost=(COMBOS[package]['price'] if package in COMBOS else 0)
    fixed_cost+=sum(max(0,DRINKS[d]-BASE_DRINK_PRICE) if package in COMBOS else DRINKS[d] for d in drinks)
    fixed_cost+=sum(MENU[k][0]*q for k,q in extras.items())
    if mode=="命運盲盒":
        b=None if budget is None or budget=="" else max(0,int(budget))
        remain=None if b is None else b-fixed_cost
        if remain is not None and remain<0:
            warnings.append(f"套餐與飲料已達 NT$ {fixed_cost}，超出盲盒預算 NT$ {-remain}；不會擅自刪除。")
        else:
            surprise=choose_blind_ingredients(direction,avoid,extras.keys()|package_items.keys(),remain,persons,seed)
            for k,q in surprise.items():extras[k]=extras.get(k,0)+q
            if not surprise and remain is not None:
                warnings.append(f"扣除套餐與飲料後剩 NT$ {remain}，無其他符合預算的盲盒食材。")
    combined=dict(package_items)
    for k,v in extras.items():combined[k]=combined.get(k,0)+v
    return {"items":combined,"package_items":package_items,"extras":extras,"drinks":drinks,
            "mode":mode,"persons":persons,"direction":direction,"package":package,
            "excluded":sorted(avoid),"warnings":warnings,
            "fixed_cost_before_blind":fixed_cost}

def calculate_price(order):
    package=order.get("package","不選套餐")
    extras=normalize_counts(order.get("extras",{}))
    package_items=normalize_counts(order.get("package_items",{}))
    details=[{"name":k,"qty":q,"unit_price":MENU[k][0],"amount":q*MENU[k][0]} for k,q in extras.items()]
    included_details=[{"name":k,"qty":q,"unit_price":MENU[k][0],"amount":q*MENU[k][0]} for k,q in package_items.items()]
    extra_total=sum(x["amount"] for x in details)
    package_price=COMBOS[package]["price"] if package in COMBOS else 0
    drink_details=[{"name":d,"qty":1,"original_price":DRINKS[d],
                    "included_value":BASE_DRINK_PRICE if package in COMBOS else 0,
                    "upgrade":max(0,DRINKS[d]-BASE_DRINK_PRICE) if package in COMBOS else 0,
                    "charge":max(0,DRINKS[d]-BASE_DRINK_PRICE) if package in COMBOS else DRINKS[d]}
                   for d in order.get("drinks",[]) if d in DRINKS]
    drink_upgrade=sum(x["upgrade"] for x in drink_details)
    drink_charge=sum(x["charge"] for x in drink_details)
    subtotal=package_price+extra_total+drink_charge
    normal_price=sum(x["amount"] for x in included_details)+len(drink_details)*BASE_DRINK_PRICE
    combo_savings=max(0,normal_price-package_price) if package in COMBOS and not order.get("warnings") else (normal_price-package_price if package in COMBOS else 0)
    return {"details":details,"included_details":included_details,"drink_details":drink_details,
            "package":package,"package_price":package_price,"extra_total":extra_total,
            "drink_upgrade":drink_upgrade,"food_total":extra_total,
            "drink_total":drink_charge,"combo_savings":combo_savings,
            "subtotal":subtotal,"items":order.get("items",{}),"warnings":order.get("warnings",[])}

def draw_surprise(subtotal):
    """僅在顧客親手選擇神秘卡時執行，並由 Gradio Session 狀態禁止重複抽獎。"""
    subtotal=max(0,int(subtotal))
    if subtotal<250:
        return {"eligible":False,"won":False,"rate":1.0,"label":"無",
                "discount":0,"message":f"🔒 消費滿 NT$250 才能抽獎，目前 NT$ {subtotal}，還差 NT$ {250-subtotal}。"}
    # 20% 未中獎；80% 中獎；中獎可得 9、8、7、6、5 折，5 折為最大獎。
    # secrets.SystemRandom() 讓同一筆選菜在重新結帳時仍有不同抽獎結果。
    prize=random.SystemRandom().choices([None,9,8,7,6,5],weights=[20,35,20,12,8,5],k=1)[0]
    if prize is None:
        return {"eligible":True,"won":False,"rate":1.0,"label":"未中獎",
                "discount":0,"message":"😆 抽獎結果：沒有中獎！本單維持原價。"}
    # 金額採四捨五入（.5 進位），避免 Python bankers rounding。
    from decimal import Decimal, ROUND_HALF_UP
    rate=Decimal(prize)/Decimal(10)
    total=int((Decimal(subtotal)*rate).quantize(Decimal('1'),rounding=ROUND_HALF_UP))
    discount=subtotal-total
    return {"eligible":True,"won":True,"rate":float(rate),"label":f"{prize}折",
            "discount":discount,"message":f"🎉 恭喜抽中 {prize} 折！折抵 NT$ {discount}。"}

def check_budget(subtotal, discount, budget):
    subtotal=max(0,int(subtotal));discount=max(0,min(subtotal,int(discount)))
    total=subtotal-discount
    if budget is None or budget=="":
        return {"subtotal":subtotal,"discount":discount,"total":total,"budget":None,"within_budget":None,"difference":None}
    budget=max(0,int(budget))
    return {"subtotal":subtotal,"discount":discount,"total":total,"budget":budget,
            "within_budget":total<=budget,"difference":budget-total}

TOOLS = {"get_menu":get_menu,"get_drink_options":get_drink_options,"build_order":build_order,"calculate_price":calculate_price,
         "draw_surprise":draw_surprise,"check_budget":check_budget}
print(f"✅ 菜單 {len(MENU)} 種食材、{len(DRINKS)} 種飲料、{len(TOOLS)} 個工具；套餐：{', '.join(COMBOS)}")

✅ 菜單 23 種食材、7 種飲料、6 個工具；套餐：單人套餐, 雙人套餐


## Step 3｜Tool Use AI Agent 與互動抽獎
結帳時 AI 使用工具組單、計價；若 AI 無法完成，由 Python 補足並在紀錄標註。**不會在結帳時偷抽獎**。完成結帳後，顧客親手選一張神秘卡，Python 才呼叫 `draw_surprise`，將結果和折扣記入同一份工具紀錄。

In [7]:
SYSTEM = """你是麻辣宇宙的 AI 店長，任務是協助點菜與計價。一次只輸出單個 JSON：
{"tool":"get_menu或get_drink_options或build_order或calculate_price或check_budget或finish","args":{}}
不得輸出 Markdown。你可以決定先查菜單或飲料，接著必須完成 build_order、calculate_price、check_budget。
務必遵守前置條件；顧客已選的食材、套餐、預算、飲料不可擅自改動。
注意：抽獎必須由顧客在結帳後親手選卡才能觸發，AI 在結帳階段絕對不可以抽獎。
所有金額由 Python 工具計算，不可自己捏造數字。
"""

def parse_action(s):
    s=re.sub(r"^```(?:json)?\s*|\s*```$","",str(s).strip())
    try: obj=json.loads(s)
    except json.JSONDecodeError:
        match=re.search(r"\{.*\}",s,re.S)
        if not match: raise ValueError("AI 沒有輸出有效 JSON")
        obj=json.loads(match.group())
    allowed={"get_menu","get_drink_options","build_order","calculate_price","check_budget","finish"}
    if not isinstance(obj,dict) or obj.get("tool") not in allowed:
        raise ValueError("AI 選擇了目前不允許的工具")
    return obj["tool"]

def next_tool(prompt):
    r=client.chat.completions.create(model=f"{provider}:{model}",
          messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],temperature=0)
    return parse_action(r.choices[0].message.content)

def make_seed(payload):
    b=hashlib.sha256(json.dumps(payload,ensure_ascii=False,sort_keys=True).encode()).digest()
    return int.from_bytes(b[:4],"big")

def make_receipt(payload,receipt,draw=None):
    package=payload["package"];mode=payload["mode"]
    subtotal=receipt["subtotal"]
    discount=draw["discount"] if draw else 0
    bill=check_budget(subtotal,discount,payload["budget"])
    lines=[f"## {'🎲' if mode=='命運盲盒' else '🍜'} {mode}｜麻辣宇宙點餐單",
           f"**用餐人數：{payload['persons']} 人｜套餐：{package}｜辣度：{payload['spicy']}**"]
    if mode=="命運盲盒":lines.append(f"🎯 盲盒方向：{payload['direction']}")
    if package in COMBOS:
        lines += ["",f"### 🍱 {package}｜優惠價 NT$ {receipt['package_price']}（含飲料）"]
        lines += [f"- {x['name']} × {x['qty']}（套餐內含）" for x in receipt['included_details']]
        lines += [f"- 🥤 {x['name']}（升級補 NT$ {x['upgrade']}）" for x in receipt['drink_details']]
        if receipt['warnings']: lines += ["⚠️ "+s for s in receipt['warnings']]
        else:lines.append(f"✅ 套餐比原價省 NT$ {receipt['combo_savings']}")
    if package not in COMBOS:
        lines += ["", "### 🥤 單點飲料"]
        lines += [f"- {x['name']} × 1：NT$ {x['charge']}" for x in receipt['drink_details']] or ["- 不點飲料"]
    lines += ["", "### 🥢 盲盒驚喜食材" if mode=="命運盲盒" else "### 🥢 自由點餐／額外加菜"]
    lines += [f"- {x['name']} × {x['qty']}（NT$ {x['unit_price']}／份）＝ NT$ {x['amount']}" for x in receipt['details']] or ["- 無額外加點"]
    if mode=="命運盲盒":
        lines.append(f"🎲 盲盒共選出 **{len(receipt['details'])} 種不同食材**，優先符合「{payload['direction']}」方向。")
    lines += ["",f"套餐優惠價：NT$ {receipt['package_price']}",
              f"食材加點：NT$ {receipt['extra_total']}",
              f"{'飲料升級差額' if package in COMBOS else '單點飲料金額'}：NT$ {receipt['drink_total']}",
              f"**抽獎前小計：NT$ {subtotal}**","","### 🎁 麻辣幸運卡"]
    if subtotal<250:
        lines.append(f"🔒 滿 NT$250 才能抽，目前還差 **NT$ {250-subtotal}**。")
    elif draw is None:
        lines.append("✅ 已解鎖！請在下方 **選一張 🌶️ 神秘辣椒卡**。抽獎只限一次，選完立即揭曉。")
    else:
        lines += [draw['message'],f"本次折扣：**{draw['label']}**｜折抵 NT$ {discount}"]
    lines.append(f"### 💰 {'抽獎後實付' if draw else '目前應付'}：NT$ {bill['total']}")
    if bill['budget'] is not None:
        if bill['within_budget']:lines.append(f"✅ 在 NT$ {bill['budget']} 預算內，剩 NT$ {bill['difference']}")
        else:lines.append(f"⚠️ 超出預算 NT$ {-bill['difference']}，不會擅自刪菜。")
    lines.append(f"🌿 蔥花：{payload['scallion']}｜香菜：{payload['cilantro']}（配料不加價）")
    if mode=="命運盲盒" and payload['avoid']:
        lines.append("🚫 盲盒已避開："+"、".join(payload['avoid']))
    if "過敏" in payload.get('request',''):
        lines.append("⚠️ 食物過敏須向實際店家確認，本模擬系統無法確保交叉接觸風險。")
    return "\n".join(lines)

def run_agent_checkout(mode, persons, package, direction, drink1, drink2, single_drink, spicy, budget, blocked, scallion, cilantro, request, *quantities):
    persons=int(persons)
    selected=({name:int(q) for name,q in zip(MENU,quantities) if q and int(q)>0}
              if mode=="正常點餐" else {})
    blocked=list(blocked or []) if mode=="命運盲盒" else []
    if mode=="命運盲盒":
        for item in MENU:
            if re.search(r"(?:不要|不吃|不能吃|避開|過敏)\s*"+re.escape(item),request or ""):
                blocked.append(item)
    payload={"mode":mode,"persons":persons,"package":package,"direction":direction,"drink1":drink1,"drink2":drink2,"single_drink":single_drink,
             "spicy":spicy,"scallion":scallion,"cilantro":cilantro,"budget":budget if mode=="命運盲盒" else None,
             "avoid":sorted(set(blocked)),"selected":selected,"request":request or ""}
    seed=make_seed(payload)
    completed={};tool_log=[];errors=[]
    def args_for(name):
        if name in ("get_menu","get_drink_options"):return {}
        if name=="build_order":return {"mode":mode,"persons":persons,"selected":selected,"direction":direction,
              "avoid":payload['avoid'],"package":package,"drink1":drink1,"drink2":drink2,"seed":seed,"single_drink":single_drink,"budget":payload["budget"]}
        if name=="calculate_price":return {"order":completed['build_order']}
        if name=="check_budget":return {"subtotal":completed['calculate_price']['subtotal'],
                                        "discount":0,"budget":payload['budget']}
    def do_tool(name,source):
        args=args_for(name);result=TOOLS[name](**args);completed[name]=result
        tool_log.append({"來源":source,"工具":name,"參數":args,"結果":result})
    dependencies={"get_menu":[],"get_drink_options":[],"build_order":[],
                  "calculate_price":["build_order"],"check_budget":["calculate_price"]}
    for turn in range(7):
        prompt="請選下一個尚未完成的工具。\n點餐需求："+json.dumps(payload,ensure_ascii=False)+"\n已完成："+json.dumps(completed,ensure_ascii=False)
        try:name=next_tool(prompt)
        except Exception as e:
            errors.append(f"AI 第 {turn+1} 步回應失敗：{type(e).__name__}: {e}");break
        if name=="finish":break
        if name in completed:continue
        if not all(dep in completed for dep in dependencies[name]):
            errors.append(f"AI 提早選擇 {name}，程式校正順序");continue
        do_tool(name,"AI 選擇")
        if name=="check_budget":break
    for name in ["build_order","calculate_price","check_budget"]:
        if name not in completed:do_tool(name,"程式補足")
    receipt=completed['calculate_price']
    trace={"使用者設定":payload,"工具執行紀錄":tool_log,
           "AI 回應異常與校正":errors,"抽獎狀態":"尚未抽獎；等待顧客選卡"}
    return payload,receipt,trace

def checkout_ui(previous,mode,persons,package,direction,drink1,drink2,single_drink,spicy,budget,blocked,scallion,cilantro,request,*quantities):
    try:
        payload,receipt,trace=run_agent_checkout(mode,persons,package,direction,drink1,drink2,single_drink,spicy,budget,blocked,scallion,cilantro,request,*quantities)
        # 相同的訂單若已抽過，重新按結帳不可以重抽。
        fingerprint=json.dumps({"payload":payload,"subtotal":receipt['subtotal']},ensure_ascii=False,sort_keys=True)
        if previous and previous.get('fingerprint')==fingerprint:
            state=previous
        else:
            state={"fingerprint":fingerprint,"payload":payload,"receipt":receipt,"trace":trace,"draw":None}
        eligible=receipt['subtotal']>=250 and state['draw'] is None
        if receipt['subtotal']<250:notice=f"🔒 還差 NT$ {250-receipt['subtotal']} 才能解鎖抽獎。"
        elif state['draw'] is None:notice="🎟️ **抽獎已解鎖！請親手選擇一張辣椒神秘卡（只能選一次）**"
        else:notice=f"🎉 本單已抽過：{state['draw']['label']}，不能再次抽獎。"
        return (make_receipt(state['payload'],state['receipt'],state['draw']),
                json.dumps(state['trace'],ensure_ascii=False,indent=2),notice,state,
                *[gr.update(interactive=eligible) for _ in range(3)])
    except Exception as e:
        return (f"⚠️ 結帳失敗：{type(e).__name__}: {e}","請檢查輸入欄位和 Groq API Key。",
                "🔒 尚未成功結帳",previous,*[gr.update(interactive=False) for _ in range(3)])

def choose_card(card_no,state):
    if not state:
        return "請先結帳。","","🔒 請先結帳",state,*[gr.update(interactive=False) for _ in range(3)]
    if state['draw'] is None and state['receipt']['subtotal']>=250:
        subtotal=state['receipt']['subtotal']
        # 重要：只在按卡片的事件中呼叫抽獎工具，沒有事先抽出結果。
        draw=draw_surprise(subtotal)
        state['draw']=draw
        state['trace']['工具執行紀錄'].append({"來源":f"顧客親手選擇第 {card_no} 張卡片",
                    "工具":"draw_surprise","參數":{"subtotal":subtotal},"結果":draw})
        state['trace']['抽獎狀態']=f"第 {card_no} 張卡片已揭曉，只能抽一次"
    draw=state['draw']
    if not draw:
        notice="🔒 尚未達到 NT$250 的抽獎門檻"
    else:
        notice=f"🎊 **你選的是第 {card_no} 張神秘卡！{draw['message']}**  本單不可重複抽獎。"
    return (make_receipt(state['payload'],state['receipt'],draw),
            json.dumps(state['trace'],ensure_ascii=False,indent=2),notice,state,
            *[gr.update(interactive=False) for _ in range(3)])


In [8]:
# ✅ 套餐、飲料升級、額外加點及抽獎測試（不用 API Key）
from unittest.mock import patch
single=build_order("正常點餐",1,{},"完全隨機",[],"單人套餐","紅茶","紅茶",42)
r=calculate_price(single)
assert r["package_price"]==139 and r["subtotal"]==139 and r["combo_savings"]==1
assert len(single["drinks"])==1 and single["drinks"]==["紅茶"]
assert sum(x["amount"] for x in r["included_details"])==115
assert r["drink_upgrade"]==0
milk=build_order("正常點餐",1,{},"完全隨機",[],"單人套餐","奶茶","紅茶",42)
assert calculate_price(milk)["subtotal"]==159  # 139 + 奶茶升級 20
both=build_order("正常點餐",2,{},"完全隨機",[],"雙人套餐","奶茶","綠茶",42)
r2=calculate_price(both)
assert r2["package_price"]==299 and r2["drink_upgrade"]==20 and r2["subtotal"]==319
assert r2["combo_savings"]==56
assert calculate_price(build_order("正常點餐",1,{"牛肉片":1},"完全隨機",[],"不選套餐","紅茶","紅茶",42))["subtotal"]==55
addon=build_order("正常點餐",1,{"毛肚":1},"完全隨機",[],"單人套餐","紅茶","紅茶",42)
assert calculate_price(addon)["subtotal"]==204
blind=build_order("命運盲盒",1,{"豆腐":1},"蔬菜滿滿",[],"不選套餐","紅茶","紅茶",42)
assert blind["mode"]=="命運盲盒" and len(blind["items"])>=1
assert draw_surprise(249)["eligible"] is False
with patch.object(random.SystemRandom,"choices",return_value=[8]):
    prize=draw_surprise(299)
assert prize["label"]=="8折" and prize["discount"]==60
assert check_budget(299,prize["discount"],None)["total"]==239
with patch.object(random.SystemRandom,"choices",return_value=[None]): miss=draw_surprise(299)
assert miss["won"] is False and miss["discount"]==0
print("✅ 通過：單人139、雙人299、免費基本飲料、奶茶升級差額、加菜、盲盒、滿額抽獎、折扣與未中獎")
# 2.9：無套餐飲料按單價算，且不會把套餐的隱藏飲料重算。
no_combo=build_order("正常點餐",1,{"豆腐":1},"完全隨機",[],"不選套餐","奶茶","奶茶",42,"冬瓜茶")
assert calculate_price(no_combo)["subtotal"]==50  # 豆腐20 + 冬瓜茶30
assert calculate_price(no_combo)["drink_total"]==30
no_drink=build_order("正常點餐",1,{"豆腐":1},"完全隨機",[],"不選套餐","奶茶","奶茶",42,"不點飲料")
assert calculate_price(no_drink)["subtotal"]==20
combo_extra=build_order("正常點餐",1,{},"完全隨機",[],"單人套餐","紅茶","紅茶",42,"奶茶")
assert calculate_price(combo_extra)["subtotal"]==139  # 隱藏的單點飲料不可計費
print("✅ 2.9 驗證：無套餐單點飲料計價、套餐含飲料、隱藏欄位不重複計費")

✅ 通過：單人139、雙人299、免費基本飲料、奶茶升級差額、加菜、盲盒、滿額抽獎、折扣與未中獎
✅ 2.9 驗證：無套餐單點飲料計價、套餐含飲料、隱藏欄位不重複計費


In [9]:
# ✅ 互動抽獎測試（無須 Groq API）
from unittest.mock import patch
sample={"mode":"正常點餐","persons":2,"package":"雙人套餐","direction":"完全隨機",
        "drink1":"紅茶","drink2":"紅茶","spicy":"微辣","scallion":"要蔥花","cilantro":"不要香菜","budget":None,"avoid":[],"selected":{},"request":""}
order=build_order("正常點餐",2,{},"完全隨機",[],"雙人套餐","紅茶","紅茶",42)
receipt=calculate_price(order)
assert receipt["subtotal"]==299
state_test={"fingerprint":"test","payload":sample,"receipt":receipt,"trace":{"工具執行紀錄":[],"抽獎狀態":"待抽"},"draw":None}
with patch.object(random.SystemRandom,"choices",return_value=[8]):
    output=choose_card(2,state_test)
assert state_test["draw"]["label"]=="8折"
assert "239" in output[0] and "draw_surprise" in output[1]
assert len(state_test["trace"]["工具執行紀錄"])==1
with patch.object(random.SystemRandom,"choices",side_effect=AssertionError("不可重抽")):
    output2=choose_card(1,state_test)
assert len(state_test["trace"]["工具執行紀錄"])==1 and state_test["draw"]["discount"]==60
print("✅ 神秘卡測試成功：NT$299 8折→NT$239；同單重點卡片不會再抽。")


✅ 神秘卡測試成功：NT$299 8折→NT$239；同單重點卡片不會再抽。


In [10]:
# ✅ 驗證抽獎機率設定與真正隨機來源（不用 Groq Key）
import inspect
source = inspect.getsource(draw_surprise)
assert "random.SystemRandom().choices" in source
assert "weights=[20,35,20,12,8,5]" in source
from unittest.mock import patch
for expected, label in [(None,"未中獎"),(9,"9折"),(8,"8折"),(7,"7折"),(6,"6折"),(5,"5折")]:
    with patch.object(random.SystemRandom, "choices", return_value=[expected]):
        result=draw_surprise(300)
        assert result["label"]==label, (expected,result)
print("✅ 六種抽獎結果全部可產生；機率：未中獎20%、9折35%、8折20%、7折12%、6折8%、5折5%")


✅ 六種抽獎結果全部可產生；機率：未中獎20%、9折35%、8折20%、7折12%、6折8%、5折5%


In [11]:
# ✅ 3.1：主食新價、盲盒完整忌口與預算、蔥花香菜與食材多樣性測試（不需 API）
assert [MENU[k][0] for k in ["王子麵","冬粉","年糕","烏龍麵","白飯"]]==[15,15,30,35,15]
assert calculate_price(build_order("正常點餐",1,{},"均衡營養感",[],"單人套餐","紅茶","紅茶",11))["combo_savings"]==1
for budget in (100,125,175,250,400):
    for direction in DIRECTIONS:
        for seed in range(20):
            bad=["魚丸","羊肉片","烏龍麵"]
            sample=build_order("命運盲盒",1,{"牛肉片":4},direction,bad,"不選套餐","紅茶","紅茶",seed,budget=budget)
            receipt=calculate_price(sample)
            assert receipt["subtotal"]<=budget,(budget,direction,seed,receipt["subtotal"])
            assert not any(k in sample["items"] for k in bad)
            compare=build_order("命運盲盒",1,{},direction,bad,"不選套餐","紅茶","紅茶",seed,budget=budget)
            assert sample["items"]==compare["items"], "盲盒不應使用手動選菜份數"
            if budget>=125:assert len(sample["extras"])>=3,(budget,direction,seed,sample)
# 排除全部食材也應正常回應
all_excluded=build_order("命運盲盒",1,{},"完全隨機",list(MENU),"不選套餐","紅茶","紅茶",1,budget=125)
assert all_excluded["items"]=={}
# 套餐價與飲料預先計入預算，低於套餐價格須提醒
combo=build_order("命運盲盒",1,{},"均衡營養感",[],"單人套餐","紅茶","紅茶",2,budget=125)
assert calculate_price(combo)["subtotal"]==139 and any("超出" in x for x in combo["warnings"])
with_drink=build_order("命運盲盒",1,{},"澱粉快樂",[],"不選套餐","紅茶","紅茶",3,"奶茶",budget=125)
assert calculate_price(with_drink)["subtotal"]<=125
print("✅ 3.1 測試通過：所有主食價格、不同方向及預算、完整排除清單、更多食材、套餐飲料成本")

✅ 3.1 測試通過：所有主食價格、不同方向及預算、完整排除清單、更多食材、套餐飲料成本


## Step 4｜Gradio 互動介面
- 不選套餐：只顯示單點飲料清單（含各飲料單價，可選不點）。
- 單人套餐：只顯示第 1 杯飲料（基本款免費、升級補差額）。
- 雙人套餐：顯示第 1、2 杯飲料（基本款皆包含）。
- 滿 NT$250 可選三張神秘卡中的一張抽獎，每筆訂單限抽一次。
- **3.1 新增**：正常點餐蔥花／香菜偏好、盲盒不吃食材；盲盒預算會先扣套餐與飲料、指定菜品，再挑選驚喜食材。
- 盲盒模式不顯示手選菜品，僅顯示所有食材的「不要清單」及蔥花香菜；正常模式保留手選菜。
- 盲盒預算自動選較多種類，並優先搭配方向。


In [ ]:
CSS="""
.gradio-container {max-width: 1260px !important;}
#hero {background:linear-gradient(110deg,#3b0a0a,#8b1919);border-radius:18px;padding:20px;color:#fff;}
#hero h1,#hero p {color:#fff !important;}
"""
with gr.Blocks(title="🌶️ 麻辣宇宙 3.1") as demo:
    gr.Markdown("# 🌶️ 麻辣宇宙 MALATANGVERSE 3.1\n### 🍜 自選點餐 × 🎲 命運盲盒 × 🥤 優惠套餐 × 🃏 辣椒神秘卡",elem_id="hero")
    gr.Markdown("**課堂展示模擬餐廳｜Tool Use Agent**：AI 選擇查菜單、組單與計價工具；顧客按卡片才觸發抽獎工具。右側可看到每一步。")
    gr.Markdown("🍱 **單人套餐 NT$139（原價140）｜雙人套餐 NT$299（原價355）**，含基本飲料，奶茶補 NT$20／杯。可自由加菜。")
    gr.Markdown("🃏 **滿 NT$250，親手選一張辣椒神秘卡**：9～5 折或沒中獎，每筆訂單只抽一次。")
    with gr.Row():
        with gr.Column(scale=5):
            mode_ui=gr.Radio(["正常點餐","命運盲盒"],value="正常點餐",label="🎮 點餐玩法")
            with gr.Row():
                persons_ui=gr.Slider(1,12,value=1,step=1,label="👥 用餐人數")
                package_ui=gr.Dropdown(["不選套餐","單人套餐","雙人套餐"],value="不選套餐",label="🍱 套餐（可再加點）")
            direction_ui=gr.Dropdown(DIRECTIONS,value="完全隨機",label="🎲 盲盒搭配方向",visible=False)
            gr.Markdown("### 🥤 飲料選擇｜依套餐顯示")
            # 無套餐：單點飲料完整單價。單人套餐：含 1 杯。雙人套餐：含 2 杯。
            with gr.Group(visible=True) as no_combo_drink_group:
                single_drink_ui=gr.Dropdown(
                    choices=[("不點飲料", "不點飲料")]+[(f"{d}｜NT$ {DRINKS[d]}",d) for d in DRINK_OPTIONS],
                    value="不點飲料",label="🥤 單點飲料（完整單價計費）")
            with gr.Group(visible=False) as one_drink_group:
                drink1_ui=gr.Dropdown(
                    choices=[(f"{d}｜{'已含' if DRINKS[d]==25 else '補 NT$ '+str(DRINKS[d]-25)}",d) for d in DRINK_OPTIONS],
                    value="紅茶",label="🥤 第 1 杯（套餐內含基本飲料）")
            with gr.Group(visible=False) as two_drink_group:
                drink2_ui=gr.Dropdown(
                    choices=[(f"{d}｜{'已含' if DRINKS[d]==25 else '補 NT$ '+str(DRINKS[d]-25)}",d) for d in DRINK_OPTIONS],
                    value="紅茶",label="🥤 第 2 杯（雙人套餐限定）")
            with gr.Row():
                spicy_ui=gr.Dropdown(["不辣","微辣","中辣","大辣","爆辣","地獄辣"],value="微辣",label="🌶️ 辣度")
                budget_ui=gr.Number(value=125,minimum=0,precision=0,label="💰 盲盒預算（留空＝無上限）",visible=False)
            with gr.Group(visible=True) as garnish_group:
                gr.Markdown("### 🌿 配料喜好（免費）")
                with gr.Row():
                    scallion_ui=gr.Radio(["要蔥花","不要蔥花"],value="要蔥花",label="蔥花")
                    cilantro_ui=gr.Radio(["要香菜","不要香菜"],value="不要香菜",label="香菜")
            with gr.Group(visible=False) as blind_exclude_group:
                blocked_ui=gr.CheckboxGroup(list(MENU),label="🚫 盲盒不要的食材（可複選）")
            gr.Markdown("🎲 **盲盒玩法：只要選搭配方向、預算和不吃食材，系統會自動挑出多種食材。**")
            with gr.Group(visible=True) as manual_food_group:
                gr.Markdown("### 🧺 正常點餐｜自己選菜品與份數（0＝不選）")
                qty_inputs=[]
                for cat in CATEGORIES:
                    with gr.Accordion(f"🥢 {cat}",open=cat in ("肉類","蔬菜")):
                        with gr.Row():
                            for food,(price,c) in MENU.items():
                                if c==cat:
                                    comp=gr.Number(value=0,minimum=0,maximum=20,precision=0,label=f"{food}｜${price}")
                                    qty_inputs.append((food,comp))
            request_ui=gr.Textbox(label="💬 給 AI 店長的備註",placeholder="例如：不要魚丸、幫我取有趣的餐點名稱",lines=2)
            go=gr.Button("🛒 確認訂單，結帳解鎖抽獎",variant="primary",size="lg")
        with gr.Column(scale=5):
            answer=gr.Markdown("## 👋 歡迎來到麻辣宇宙！\n先選菜結帳，滿 NT$250 才能抽辣椒神秘卡。")
            gr.Markdown("### 🃏 選你的命運辣椒卡")
            notice=gr.Markdown("🔒 尚未結帳，神秘卡未解鎖。")
            with gr.Row():
                card1=gr.Button("🌶️ 神秘卡 ①",interactive=False)
                card2=gr.Button("🌶️ 神秘卡 ②",interactive=False)
                card3=gr.Button("🌶️ 神秘卡 ③",interactive=False)
            gr.Markdown("*抽獎機率：未中獎 20%、9折 35%、8折 20%、7折 12%、6折 8%、5折 5%。卡片選擇是參與儀式，不會影響中獎率。*")
            log=gr.Code(label="🛠️ AI Tool Use 工具紀錄（老師重點）",language="json",lines=22)
    state=gr.State(None)
    ordered_inputs=[dict(qty_inputs)[k] for k in MENU]
    go.click(fn=checkout_ui,
        inputs=[state,mode_ui,persons_ui,package_ui,direction_ui,drink1_ui,drink2_ui,single_drink_ui,spicy_ui,budget_ui,blocked_ui,scallion_ui,cilantro_ui,request_ui]+ordered_inputs,
        outputs=[answer,log,notice,state,card1,card2,card3])
    for no,btn in [(1,card1),(2,card2),(3,card3)]:
        btn.click(fn=lambda s,n=no:choose_card(n,s),inputs=[state],outputs=[answer,log,notice,state,card1,card2,card3])
    def set_mode(mode):
        blind=mode=="命運盲盒"
        return (gr.update(visible=blind),gr.update(visible=blind),
                gr.update(visible=blind),gr.update(visible=not blind))
    mode_ui.change(fn=set_mode,inputs=[mode_ui],outputs=[direction_ui,budget_ui,blind_exclude_group,manual_food_group])
    def show_drinks(package):
        return (gr.update(visible=package=="不選套餐"),
                gr.update(visible=package in ("單人套餐","雙人套餐")),
                gr.update(visible=package=="雙人套餐"))
    package_ui.change(fn=show_drinks,inputs=[package_ui],
                      outputs=[no_combo_drink_group,one_drink_group,two_drink_group])
    gr.Markdown("📌 **展示建議：** 選雙人套餐 NT$299 → 結帳 → 親手點一張卡 → 出現折扣或未中獎 → 查看右側 `draw_surprise` 工具紀錄。")
    gr.Markdown("⚠️ 模擬店鋪、模擬價格與趣味抽獎，非實際銷售。")
demo.launch(share=True,debug=True,show_error=True,theme=gr.themes.Soft(),css=CSS)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c5690b9ac8ba707d70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 5｜作業說明、測試與改進紀錄

### Agent 設計方法：Tool Use
AI 店長透過 Groq 模型判斷要使用哪個 Python 工具查菜單、組單及計價。工具執行順序、參數、結果全部記錄。當 AI 沒有正確完成工具鏈，程式會補足並**明確標示「程式補足」**。抽獎工具則由顧客按卡片觸發，不假裝由 AI 自動選擇。

### 2.5 → 2.6 改良重點
1. 以前按結帳時自動抽獎，看不到顧客參與過程。
2. 新增「三張辣椒神秘卡」，滿 NT$250 才能點；**點選時才抽**，讓使用者有參與感。
3. 同筆訂單只允許抽一次，避免一直重新按結帳重抽。
4. 抽完顯示折扣、折抵金額和最後實付，右側新增抽獎工具紀錄。
5. 保留單人139元／雙人299元的套餐優惠、飲料與自由加點。

### 建議截圖
- 單人套餐139元：不能抽獎。
- 雙人套餐299元：三張卡可選。
- 顧客選卡後：顯示結果、實付及工具紀錄中的 `draw_surprise`。
- 重新結帳相同訂單：不能重抽。

### 模擬抽獎機率
未中獎50%；9折20%；8折12%；7折9%；6折6%；5折3%。

本作業模擬點餐與優惠，不代表真實餐廳活動。
### 2.7 排錯紀錄
- 修正雙人套餐飲料欄位可能因前端切換事件未執行而隱藏：兩杯飲料選單固定顯示；選單人套餐只計第一杯，選雙人套餐才計兩杯。
- Gradio 6 相容性：將 `theme`、`css` 放到 `launch()`。
- `Unexpected token <` 通常表示 Gradio 網頁收到 HTML 而不是預期的 JSON（例如使用已失效的舊公開連結或服務暫時中斷）；應以最新 Colab 執行輸出的網址為準。


### 3.1 新增測試與改良
- 主食改價：王子麵／冬粉／白飯 NT$15，年糕 NT$30，烏龍麵 NT$35。
- 單人套餐原價因王子麵調價，現在是 NT$140；套餐價 NT$139，省 NT$1。
- 盲盒先計入套餐、飲料及客人指定菜品，再依剩餘預算加菜，不故意超支；若必選品項已超支，明確告知。
- 隱藏的「不要食材」只屬於命運盲盒；正常點餐提供免費蔥花、香菜偏好。

### 3.1 改善記錄
- 盲盒改為不手選食材；改以整份食材清單逐項排除。
- 正常點餐與盲盒都可選蔥花、香菜。
- 優先選更多種不同食材，依搭配方向加權挑選，最終付款不得超過可用預算（若套餐本身超出則明示）。
